# Basic usage of colzette

In [ ]:
import time
from pathlib import Path # deal with paths in python 3

import pandas

from openalea.colzette.colzette import compute_thermal_time, df_to_dict, run_static_simulation


In [ ]:
root_project_dir = Path('.').absolute().parent
data_dir = root_project_dir / 'data'

## Define identifiers for simulation

In [ ]:
species = 'Rapeseed' # Rapeseed or Fababean
Type_simul = "intercrop_aviso_RRF"

## Read climate file for the chosen simulation site

In [ ]:
meteo_fn = data_dir / 'climate' / 'IDEEV_2023_INRAE_STATION_91272001.csv'
clim= pandas.read_csv(meteo_fn,sep=";",skiprows=11)
clim['TM'] = (clim['TN'] + clim['TX'])/2

idx_das = clim.index[
    (clim["AN"] == 2023) &
    (clim["MOIS"] == 9) &
    (clim["JOUR"] == 20)][0] # rapeseed sowing date 14/09/2023
clim2 = clim.iloc[idx_das:]
clim2['TT'] = compute_thermal_time(vec_temp=clim2['TM'], idx_begin=0, Tb=5)

## Read Total Leaf Area data (TLA) for the chosen simulation site

In [ ]:
TLA_fn = data_dir / 'input_leaf_surface' / 'set_TLA_IDEEV.csv'
TLA_all = pandas.read_csv(TLA_fn,sep="\t")
TLA_indices_keep = TLA_all[TLA_all['Type']==Type_simul].index
vec_TLA = TLA_all['sim'].values[TLA_indices_keep]

## Read parameters based on option (default or specific to an experimental treatment)

In [ ]:
dict_params = df_to_dict(data_dir,Type_simul,Type_simul,"")
density=33.0

## run simulation over the whole clim data

In [ ]:
dfs = []
list_scene = []
option_plants = 'plot' # 'plot' or 'single' i.e. crop or one plant
for iday in range(0,len(clim2)):
    print(iday)
    PlantAge = clim2.iloc[iday]['TT']
    RG_daily = clim2.iloc[iday]['PAR']
    TLA = vec_TLA[iday]
    scene, sub_dat = run_static_simulation(iday,
                                  PlantAge,
                                  RG_daily,
                                  TLA,
                                  option_plants,
                                  density,
                                  dict_params,
                                  type_simul=Type_simul,
                                  species=species)
    dfs.append(sub_dat)
    list_scene.append(scene)

df = pandas.concat(dfs,ignore_index=True)